# csp_workflow_mp — end-to-end example (KTaO$_3$)

This notebook walks a chemical formula through the full pipeline:

1. Compute the 36-dim periodic descriptor from the formula.
2. Predict the top-K space groups with the trained XGBoost classifier.
3. Retrieve MP structure templates whose space group matches the classifier's top-1 prediction.
4. Substitute the target elements onto the top template.
5. Relax the substituted structure with MatterSim (BFGS + UnitCellFilter).
6. Compare the relaxed structure against the MP reference.

Target: **KTaO$_3$** — a cubic perovskite (Pm-3m, space group 221). It is fully ordered, has only three elements, and is a textbook high-permittivity dielectric, which makes it a clean single-composition demo.

## Prerequisites

This notebook uses real MP data and a real trained classifier. Before running it, complete steps 1–3 of the paper reproduction pipeline (see the README):

```bash
python scripts/01_download_mp_data.py     # needs MP_API_KEY, ~5 GB
python scripts/02_compute_descriptors.py
python scripts/03_train_xgboost.py         # ~40 min
```

The relaxation step needs MatterSim installed (`pip install -e ".[relaxation]"`) and works fastest on a CUDA GPU (~30 s / structure). CPU is also fine (~2 min / structure).

## 1. Encode the formula as a periodic descriptor

In [1]:
from csp_workflow_mp import (
    compute_periodic_descriptors,
    predict_top_k_space_groups,
    TemplatePool,
    SubstitutionEngine,
)

target = 'KTaO3'
desc = compute_periodic_descriptors(target)
print(f'target formula   : {target}')
print(f'descriptor shape : {desc.shape}   (18 group fractions + 18 mean atomic numbers)')

target formula   : KTaO3
descriptor shape : (36,)   (18 group fractions + 18 mean atomic numbers)


## 2. Predict the top-K space groups

`predict_top_k_space_groups` lazy-loads the trained XGBoost model on first call.
The paper's primary retrieval mode is K = 1 (the single most confident SG prediction);
K = 3 is shown here for comparison and matches the practical coverage–fidelity setting
discussed in the Discussion.


In [2]:
top_1 = predict_top_k_space_groups(desc, k=1)
top_3 = predict_top_k_space_groups(desc, k=3)
print(f'top-1 predicted SG : {top_1}')
print(f'top-3 predicted SG : {top_3}')
print()
print(f'For KTaO3 the ground-truth space group in MP is 221 (Pm-3m cubic perovskite).')
print(f'A correct top-1 prediction means K=1 retrieval will directly hit that family.')

top-1 predicted SG : [221]
top-3 predicted SG : [221, 62, 38]

For KTaO3 the ground-truth space group in MP is 221 (Pm-3m cubic perovskite).
A correct top-1 prediction means K=1 retrieval will directly hit that family.


## 3. Retrieve MP templates matching the predicted space group

`TemplatePool.search` filters the ~103 K MP pool to entries whose space group
matches the predicted SG and ranks the remaining candidates by descriptor cosine
distance to the KTaO$_3$ descriptor.

In [3]:
pool = TemplatePool(
    'data/MP/metadata_with_descriptors.csv',
    cif_root='data/MP/cifs',
    exclude_ids={'mp-3614'},   # exclude KTaO3 itself (LOEO convention)
)
print(f'pool size: {len(pool)}')

hits = pool.search(space_group=top_1[0], descriptor_vector=desc, top_n=10)
hits[['material_id', 'formula', 'space_group_number', 'pd_distance']].head(10)

pool size: 103166


,material_id,formula,space_group_number,pd_distance
0,mp-997591,CsNa7Ta8O24,221,0.000517
1,mp-3136,NaNbO3,221,0.003296
2,mp-997599,Na8Ta7VO24,221,0.004186
3,mp-4170,NaTaO3,221,0.005454
4,mp-935811,KNbO3,221,0.018135
5,mp-1076534,RbTaO3,221,0.022711
6,mp-2311,NbO,221,0.035303
7,mp-1187250,TaTi3,221,0.078650
8,mp-20572,NaVF3,221,0.079700
9,mp-11358,TaCo3,221,0.097471


## 4. Substitute the target formula onto the top template

`SubstitutionEngine.find_substitutions` enumerates chemically valid element
mappings between the target composition and the template, using the chemical-role
assignment table from the paper (Table S5). We take the first feasible mapping,
which is one-to-one (K→A-site, Ta→B-site, O→X-site of the perovskite) for KTaO$_3$.

In [4]:
from pymatgen.core import Structure
from pathlib import Path

top = hits.iloc[0]
template_cif = Path('data/MP/cifs') / f"{top['material_id']}.cif"
template_struct = Structure.from_file(str(template_cif))
print(f'top template : {top["material_id"]}  {top["formula"]}  SG={top["space_group_number"]}')

engine  = SubstitutionEngine()
results = engine.find_substitutions(target, template_struct)
feasible = [r for r in results if r.success]
print(f'feasible substitution mappings: {len(feasible)} / {len(results)}')

pred = engine.apply_substitution(template_struct, feasible[0])
print()
print(f'template   : {template_struct.composition.reduced_formula}   ({len(template_struct)} sites)')
print(f'predicted  : {pred.composition.reduced_formula}   ({len(pred)} sites)')

top template : mp-997591  CsNa7Ta8O24  SG=221
feasible substitution mappings: 1 / 1

template   : CsNa7Ta8O24   (40 sites)
predicted  : KTaO3   (40 sites)


C:\ProgramData\miniforge3\envs\csp\lib\_collections_abc.py:824: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  return self[key]


## 5. Relax the predicted structure with MatterSim

The substituted structure inherits the template's lattice constants and atomic positions verbatim; only the chemical identities have changed. Relaxation with MatterSim then relaxes both cell and atomic coordinates simultaneously (BFGS + `UnitCellFilter`, `fmax = 0.05` eV/Å, up to 500 steps).

**MatterSim requires ordered structures.** Because our substitution is fully one-to-one for KTaO$_3$, the predicted structure is already ordered and can be relaxed directly. For targets that end up partial-occupancy after substitution, MatterSim cannot relax them; the paper's benchmark (`scripts/05_run_benchmark.py`) excludes such candidates from the relaxation step and counts them as substitution failures on the valid subset (paper Methods §4.5). See `notebooks/03_partial_occupancy_handling.ipynb` for the corresponding behaviour of the high-level `predict_from_formula` API.

In [5]:
import warnings
warnings.filterwarnings('ignore')

from mattersim.forcefield import MatterSimCalculator
from pymatgen.io.ase import AseAtomsAdaptor
from ase.optimize import BFGS
try:
    from ase.filters import UnitCellFilter
except ImportError:
    from ase.constraints import UnitCellFilter

# Auto-select CUDA if available, else MPS (Apple Silicon), else CPU.
import torch
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'MatterSim device: {device}')

calc    = MatterSimCalculator(device=device)
adaptor = AseAtomsAdaptor()
atoms   = adaptor.get_atoms(pred)
atoms.calc = calc

vol_pre = atoms.get_volume()
opt = BFGS(UnitCellFilter(atoms), logfile=None)
opt.run(fmax=0.05, steps=500)
vol_post = atoms.get_volume()

relaxed = adaptor.get_structure(atoms)
print(f'relaxation converged : {opt.converged()}')
print(f'volume change |ΔV/V| : {abs(vol_post - vol_pre)/vol_pre * 100:.2f} %')
print(f'lattice parameter a  : {relaxed.lattice.a:.4f} Å')

MatterSim device: cuda
2026-07-15 18:20:25.999 | INFO     | mattersim.forcefield.potential:from_checkpoint:877 - Loading the pre-trained mattersim-v1.0.0-1M.pth model


relaxation converged : True
volume change |ΔV/V| : 2.27 %
lattice parameter a  : 8.0635 Å


## 6. Compare the relaxed prediction against the MP reference

The paper uses three complementary metrics: (i) integer space-group match, (ii)
SOAP local-environment cosine similarity, and (iii) RMSD on the ordered subset.
Here we show the primary metric — space-group match — plus the RMSD from
`pymatgen.StructureMatcher`.

In [6]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.structure_matcher import StructureMatcher

ref = Structure.from_file('data/MP/cifs/mp-3614.cif')
ref_sg   = SpacegroupAnalyzer(ref,     symprec=0.1, angle_tolerance=5).get_space_group_number()
pred_sg  = SpacegroupAnalyzer(relaxed, symprec=0.1, angle_tolerance=5).get_space_group_number()
print(f'reference SG : {ref_sg}')
print(f'predicted SG : {pred_sg}     SG match: {pred_sg == ref_sg}')

# RMSD from pymatgen.StructureMatcher (used purely as an RMSD calculator;
# the paper does not report the boolean "StructureMatcher match" as a metric).
matcher = StructureMatcher(ltol=0.2, stol=0.3, angle_tol=5)
rms_result = matcher.get_rms_dist(ref, relaxed)
if rms_result is not None:
    rms, _ = rms_result
    avg_a  = (ref.lattice.a + relaxed.lattice.a) / 2
    rmsd_A = float(rms) * avg_a
    print(f'RMSD         : {rmsd_A:.4f} Å')
else:
    print('RMSD         : (no correspondence found by StructureMatcher)')

reference SG : 221
predicted SG : 221     SG match: True
RMSD         : 0.0000 Å


## 7. Save the predicted structure

For downstream use, the relaxed candidate can be written back out as a CIF.

In [7]:
out_cif = Path('KTaO3_predicted.cif')
relaxed.to(filename=str(out_cif))
print(f'wrote {out_cif}  ({out_cif.stat().st_size} bytes)')

wrote KTaO3_predicted.cif  (2789 bytes)


## 8. The full pipeline in one call: `predict_from_formula`

`predict_from_formula` wraps Sections 1–5 (descriptor → classifier → retrieval → substitution → relaxation) into a single call and saves both the substituted and relaxed CIFs to `output_dir` automatically. Two common use cases:

- **Standard**: let the XGBoost classifier pick the space group.
- **User already knows the space group** (from XRD, Rietveld refinement, or literature): pass `known_sg=` to skip the classifier and filter the template pool directly by that SG.

In [8]:
from csp_workflow_mp import predict_from_formula

# Path A: standard — classifier picks the SG
result_A = predict_from_formula(
    'KTaO3',
    top_k_sg=1,
    n_retry=20,
    output_dir='predictions_demo/classifier',
    do_relax=True,
    verbose=False,
)
print('=== Path A: classifier picks SG ===')
print(result_A.summary())

=== Path A: classifier picks SG ===
target: KTaO3
status: SUCCESS
space group(s): [221] (classifier top-K)
template: mp-557257 (KVF3) at rank 6
substitution method: one_to_one
is_ordered: True
substituted CIF: predictions_demo\classifier\KTaO3_substituted.cif
relaxed CIF: predictions_demo\classifier\KTaO3_relaxed.cif
|ΔV/V|: 0.068
predicted SG: 221


In [9]:
# Path B: user knows the SG — skip classifier
result_B = predict_from_formula(
    'KTaO3',
    known_sg=221,        # ← skip classifier entirely
    n_retry=20,
    output_dir='predictions_demo/known_sg',
    do_relax=True,
    verbose=False,
)
print('=== Path B: user-specified known_sg=221 ===')
print(result_B.summary())

=== Path B: user-specified known_sg=221 ===
target: KTaO3
status: SUCCESS
space group: 221 (user-specified)
template: mp-557257 (KVF3) at rank 6
substitution method: one_to_one
is_ordered: True
substituted CIF: predictions_demo\known_sg\KTaO3_substituted.cif
relaxed CIF: predictions_demo\known_sg\KTaO3_relaxed.cif
|ΔV/V|: 0.068
predicted SG: 221


For KTaO3 the classifier's top-1 SG is already 221, so both paths converge on the same template. `known_sg` is most useful when the classifier is uncertain or when strong prior knowledge exists.

## What next?

- To reproduce this pipeline on your own target, just change `target = 'KTaO3'`
  at the top of section 1.
- To sweep multiple candidates at once, loop `predict_from_formula` (Section 8)
  over a composition list:

  ```python
  from csp_workflow_mp import predict_from_formula
  for f in ['KTaO3', 'NaTaO3', 'LiTaO3']:
      r = predict_from_formula(f, top_k_sg=1, do_relax=True, verbose=False)
      print(f'{f:10s} {r.status:14s} → template {r.template_material_id}')
  ```
- To benchmark systematically over 500 targets, see the reproduction pipeline
  in the README and `scripts/05_run_benchmark.py`.